In [2]:
import os
import pandas as pd
import sys
import pysam
import tqdm
from Bio import Seq
from functools import reduce
import numpy as np
import RNA
from collections import defaultdict
from tabulate import tabulate
from itertools import combinations
from collections import defaultdict

mirtar_dir = '/Users/rflusi/Library/CloudStorage/OneDrive-Stanford/seq/RNAseq/ref-dbs/miRtarBase'
tgt_sites_path = os.path.join(mirtar_dir, 'MicroRNA_Target_Sites.csv.gz')
# gene_out_path = 
# transcript_out_path = 

In [ ]:
tgt_sites_df = pd.read_csv(tgt_sites_path, index_col=)
display(pd.read_csv(tgt_sites_path))

,miRTarBase ID,miRNA,Species (miRNA),Target Gene,Target Gene (Entrez ID),Species (Target Gene),Target Site,Experiments,Support Type,References (PMID)
0,MIRT003135,mmu-miR-122-5p,mmu,Cd320,54219.0,mmu,AGAGACTGGGGTCCTCAGACACTCCC,Luciferase reporter assay//qRT-PCR//Western blot,Functional MTI,18158304.0
1,MIRT003135,mmu-miR-122-5p,mmu,Cd320,54219.0,mmu,GGGGCTACTGTGGTTTAGACACTCCC,Luciferase reporter assay//qRT-PCR//Western blot,Functional MTI,18158304.0
2,MIRT003134,mmu-miR-122-5p,mmu,Ndrg3,29812.0,mmu,GAAGAAAAGGTTTTTCACACACTCCG,Luciferase reporter assay//qRT-PCR//Western blot,Functional MTI,18158304.0
3,MIRT003134,mmu-miR-122-5p,mmu,Ndrg3,29812.0,mmu,GCTCGCCCGATCACGAACCCACTCCA,Luciferase reporter assay//qRT-PCR//Western blot,Functional MTI,18158304.0
4,MIRT003134,mmu-miR-122-5p,mmu,Ndrg3,29812.0,mmu,TCCCTCTCCTTGAAATGACCACTCCA,Luciferase reporter assay//qRT-PCR//Western blot,Functional MTI,18158304.0
...,...,...,...,...,...,...,...,...,...,...
746055,MIRT568625,hsa-miR-424-5p,hsa,ACVR2A,92.0,hsa,AACACAUAAAAUGCAGCUGCUAU//GGACUCUGAACUGGAGCUGCUAA,Luciferase reporter assay//qRT-PCR//Immunoprec...,Functional MTI,38851743.0
746056,MIRT756476,hsa-miR-136-3p,hsa,BMPR1B,658.0,hsa,GAGUAAGAUAUUAUGGAUGAUAA//GAGUAAGAUAUUAUGGAUGAUAA,Luciferase reporter assay//qRT-PCR//Immunoprec...,Functional MTI,38851743.0
746057,MIRT756477,hsa-miR-139-5p,hsa,BMPR1A,657.0,hsa,UGUACAUACAUUCAUACUGUAGA//UGUACAUACAUUCAUACUGUAGA,Luciferase reporter assay//qRT-PCR//Immunoprec...,Functional MTI,38851743.0
746058,MIRT756478,hsa-miR-223-3p,hsa,ACVR2A,92.0,hsa,AAUGGAGUGUUUGAA-AACUGA CA//AAUGGAGUGUUUGAA-AAC...,Luciferase reporter assay//qRT-PCR//Immunoprec...,Functional MTI,38851743.0


In [2]:
def import_biomart(path):
    rename_dict = {
        'Gene stable ID':'gene_id',
        'Transcript stable ID':'transcript_id',
        'Protein stable ID':'protein_id',
        'Protein stable ID version':'protein_id_version',
        'Exon stable ID':'exon_id',
        'Gene start (bp)':'gene_start',
        'Gene end (bp)':'gene_end',
        'Strand':'strand',
        'Transcript start (bp)':'transcript_start',
        'Transcript end (bp)':'transcript_end',
        'Transcription start site (TSS)':'transcription_start',
        'Transcript length (including UTRs and CDS)':'transcript_len',
        'Transcript support level (TSL)':'transcript_support',
        'GENCODE basic annotation':'GENCODE_basic',
        'GENCODE primary annotation':'GENCODE_primary',
        'RefSeq match transcript (MANE Select)':'refseq_match_transcript_mane_select',
        'RefSeq match transcript (MANE Plus Clinical)':'refseq_match_transcript_mane_plus_clinical',
        'Gene name':'gene_name',
        'Transcript name':'transcript_name',
        'Transcript count':'transcript_count',
        'Gene type':'gene_type',
        'Transcript type':'transcript_type',
        'NCBI gene (formerly Entrezgene) ID':'gene_id_entrez',
    }

    dtype_dict ={
        'Gene stable ID':str,
        'Transcript stable ID':str,
        'Protein stable ID':str,
        'Protein stable ID version':str,
        'Exon stable ID':str,
        'Gene start (bp)':int,
        'Gene end (bp)':int,
        'Strand':int,
        'Transcript start (bp)':int,
        'Transcript end (bp)':int,
        'Transcription start site (TSS)':int,
        'Transcript length (including UTRs and CDS)':int,
        'Transcript support level (TSL)':str,
        'GENCODE basic annotation':str,
        'GENCODE primary annotation':str,
        'RefSeq match transcript (MANE Select)':str,
        'RefSeq match transcript (MANE Plus Clinical)':str,
        'Gene name':str,
        'Transcript name':str,
        'Transcript count':int,
        'Gene type':str,
        'Transcript type':str,
        'NCBI gene (formerly Entrezgene) ID':str,
    }

    na_val_dict = {c:[''] for c, dtype in dtype_dict.items() if dtype == str}

    keep_cols = [
        'gene_id',
        'transcript_id',
        'protein_id',
        'exon_id',
        'gene_start',
        'gene_end',
        'strand',
        'transcript_start',
        'transcript_end',
        'transcription_start',
        'transcript_len',
        'transcript_support',
        'GENCODE_basic',
        'GENCODE_primary',
        'refseq_match_transcript_mane_select',
        'refseq_match_transcript_mane_plus_clinical',
        'gene_name',
        'transcript_name',
        'transcript_count',
        'gene_type',
        'transcript_type',
        'gene_id_entrez',
    ]

    df = pd.read_csv(
            path, sep='\t', compression='infer',
            dtype=dtype_dict, na_values=na_val_dict
        )
    
    for col in df.columns:
        if dtype_dict[col] == str:
            df.loc[df[col].apply(pd.isna), col] = ''
            df[col] = df[col].astype(str)

    df = df.rename(columns=rename_dict)

    return df[keep_cols]

def collapse_biomart(path, collapse_to):
    # -------------------------------------------------
    # Input:    1. path - path to raw biomart export
    #           2. collapse_to - list of columns to group by                
    # Output:   a df of humn mirtar base grouped by collapse_to
    # -------------------------------------------------
    df = import_biomart(path)

    group_by = df.groupby(collapse_to)
    non_groupby_cols = pd.Series(df.columns)[~(pd.Series(df.columns).isin(collapse_to))].to_list()
    col_sets = group_by[non_groupby_cols[0]].apply(lambda x: [val for val in x.unique() if not pd.isna(val)])
    grouped_df = pd.DataFrame(col_sets)

    for col in non_groupby_cols[1:]:
        col_sets = group_by[col].apply(lambda x: 
                [
                    val for val in x.unique()
                    if not (pd.isna(val) or (val == '')) 
                ]
            )
        grouped_df[col] = col_sets
    
    for col in grouped_df.columns:
        if (grouped_df[col].apply(len) == 1).all():
            grouped_df[col] = grouped_df[col].apply(lambda x: x[0])
        else:
            grouped_df[col] = grouped_df[col].apply(lambda x: ','.join([str(v) for v in x])).astype(str)

    return grouped_df

In [3]:
# gene level df 

df = collapse_biomart(biomart_path, collapse_to=['gene_id'])

keep_cols = [
        'transcript_id',
        'protein_id',
        'exon_id',
        'gene_start',
        'gene_end',
        'strand',
        'gene_name',
        'transcript_count',
        'gene_type',
        'gene_id_entrez',
    ]

df = df[keep_cols]
df.to_parquet(gene_out_path, compression='gzip')

display(df)

,transcript_id,protein_id,exon_id,gene_start,gene_end,strand,gene_name,transcript_count,gene_type,gene_id_entrez
gene_id,,,,,,,,,,
ENSG00000000003,"ENST00000373020,ENST00000612152,ENST0000049677...","ENSP00000362111,ENSP00000482130,ENSP0000053794...","ENSE00001855382,ENSE00003662440,ENSE0000365457...",100627108,100639991,-1,TSPAN6,13,protein_coding,7105
ENSG00000000005,"ENST00000373031,ENST00000485971",ENSP00000362122,"ENSE00001459371,ENSE00000401061,ENSE0000067340...",100584936,100599885,1,TNMD,2,protein_coding,64102
ENSG00000000419,"ENST00000466152,ENST00000371582,ENST0000068304...","ENSP00000507119,ENSP00000360638,ENSP0000050698...","ENSE00003522354,ENSE00003652000,ENSE0000367109...",50934852,50959140,-1,DPM1,22,protein_coding,8813
ENSG00000000457,"ENST00000367771,ENST00000367770,ENST0000042367...","ENSP00000356745,ENSP00000356744,ENSP0000040799...","ENSE00003849588,ENSE00003656990,ENSE0000358931...",169846981,169894267,-1,SCYL3,13,protein_coding,57147
ENSG00000000460,"ENST00000498289,ENST00000472795,ENST0000049697...","ENSP00000490194,ENSP00000490333,ENSP0000048973...","ENSE00001920509,ENSE00001834890,ENSE0000181194...",169662007,169855456,1,FIRRM,22,protein_coding,55732
...,...,...,...,...,...,...,...,...,...,...
ENSG00000310589,ENST00000851019,,"ENSE00004283415,ENSE00004283416,ENSE0000428341...",33140606,33155824,1,,1,transcribed_unitary_pseudogene,
ENSG00000310590,ENST00000851027,ENSP00000521096,ENSE00004283438,84101970,84101996,-1,,1,protein_coding,
ENSG00000310591,ENST00000851058,,"ENSE00004283597,ENSE00004283598",77629671,77634460,1,OTP-AS1,1,lncRNA,139225791


In [4]:
# transcript level df 

df = collapse_biomart(biomart_path, collapse_to=['transcript_id'])

keep_cols = [
        'gene_id',
        'protein_id',
        'exon_id',
        'gene_start',
        'gene_end',
        'strand',
        'transcript_start',
        'transcript_end',
        'transcription_start',
        'transcript_len',
        'transcript_support',
        'GENCODE_basic',
        'GENCODE_primary',
        'refseq_match_transcript_mane_select',
        'refseq_match_transcript_mane_plus_clinical',
        'gene_name',
        'transcript_name',
        'transcript_count',
        'gene_type',
        'transcript_type',
        'gene_id_entrez',
    ]

df = df[keep_cols]
df.to_parquet(transcript_out_path, compression='gzip')

display(df)

,gene_id,protein_id,exon_id,gene_start,gene_end,strand,transcript_start,transcript_end,transcription_start,transcript_len,...,GENCODE_basic,GENCODE_primary,refseq_match_transcript_mane_select,refseq_match_transcript_mane_plus_clinical,gene_name,transcript_name,transcript_count,gene_type,transcript_type,gene_id_entrez
transcript_id,,,,,,,,,,,,,,,,,,,,,
ENST00000000233,ENSG00000004059,ENSP00000000233,"ENSE00003494180,ENSE00003504066,ENSE0000367897...",127588370,127591700,1,127588411,127591700,127588411,1032,...,GENCODE basic,GENCODE primary,NM_001662.4,,ARF5,ARF5-201,10,protein_coding,protein_coding,381
ENST00000000412,ENSG00000003056,ENSP00000000412,"ENSE00001348389,ENSE00003523177,ENSE0000363124...",8940358,8951856,-1,8940361,8949645,8949645,2450,...,GENCODE basic,GENCODE primary,NM_002355.4,,M6PR,M6PR-201,23,protein_coding,protein_coding,4074
ENST00000000442,ENSG00000173153,ENSP00000000442,"ENSE00003589560,ENSE00003471121,ENSE0000072724...",64305080,64316744,1,64305524,64316743,64305524,2274,...,GENCODE basic,GENCODE primary,NM_004451.5,,ESRRA,ESRRA-201,34,protein_coding,protein_coding,2101
ENST00000001008,ENSG00000004478,ENSP00000001008,"ENSE00000802791,ENSE00003687749,ENSE0000378412...",2794290,2805423,1,2794970,2805423,2794970,3715,...,GENCODE basic,GENCODE primary,NM_002014.4,,FKBP4,FKBP4-201,19,protein_coding,protein_coding,2288
ENST00000001146,ENSG00000003137,ENSP00000001146,"ENSE00000401861,ENSE00000846593,ENSE0000362377...",72129238,72147862,-1,72129238,72147862,72147862,4556,...,GENCODE basic,GENCODE primary,NM_019885.4,,CYP26B1,CYP26B1-201,5,protein_coding,protein_coding,56603
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENST00000973043,ENSG00000292357,ENSP00000642905,"ENSE00004014975,ENSE00004014946,ENSE0000401494...",1268779,1325373,1,1268814,1309887,1268814,2021,...,GENCODE basic,,,,CSF2RA,CSF2RA-266,34,protein_coding,protein_coding,1438
ENST00000973044,ENSG00000292357,ENSP00000642906,"ENSE00004014946,ENSE00004014947,ENSE0000401494...",1268779,1325373,1,1268825,1309887,1268825,1855,...,GENCODE basic,,,,CSF2RA,CSF2RA-267,34,protein_coding,protein_coding,1438
ENST00000973045,ENSG00000292357,ENSP00000642907,"ENSE00004014946,ENSE00004014947,ENSE0000401494...",1268779,1325373,1,1274765,1309887,1274765,1935,...,GENCODE basic,,,,CSF2RA,CSF2RA-268,34,protein_coding,protein_coding,1438
